# NetMHCpan / PRIME / IEDB vs Ensemble on the External Cohort



In [ ]:
!pip install rapidfuzz

## Step 1: Create the script
The cell below creates the `benchmark_tools.py` file (version 2) in Colab.

In [ ]:
import argparse
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler

AA = set("ACDEFGHIKLMNPQRSTVWY")
# The 6 class I alleles used when the dataset was built
CLASS_I_ALLELES = ["HLA-A*01:01", "HLA-A*02:01", "HLA-A*03:01",
                   "HLA-A*24:02", "HLA-B*07:02", "HLA-B*44:03"]
URL_PRIME = "https://github.com/GfellerLab/PRIME.git"
URL_MIX = "https://github.com/GfellerLab/MixMHCpred.git"


def prime_allele(a: str) -> str:
    """HLA-A*02:01 -> A0201"""
    return a.replace("HLA-", "").replace("*", "").replace(":", "")


def fast_auc(y, s):
    """ROC-AUC by the rank (Mann-Whitney) formula; ties get average ranks (same value as sklearn)."""
    y = np.asarray(y); s = np.asarray(s, float)
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def resolve_file(path, what):
    """Return an existing path. Colab renames re-uploads ('name (1).csv'), so one close match is accepted;
    otherwise stop with a clear message instead of a traceback."""
    if not path:
        return path
    p = Path(path)
    if p.exists():
        return str(p)
    cands = sorted(p.parent.glob(p.stem + "*" + p.suffix))
    if len(cands) == 1:
        print(f"NOTE: '{path}' not found, using '{cands[0]}' instead")
        return str(cands[0])
    here = sorted(x.name for x in Path(".").glob("*.csv"))
    sys.exit(f"FILE NOT FOUND: '{path}'  ({what})\n"
             f"CSV files in the current folder: {here if here else 'none'}\n"
             f"Upload it again with the upload cell (Colab deletes uploaded files when the runtime restarts), "
             f"or fix the file name in the command.")


def _run(cmd, cwd=None, shell=False, check=True):
    print("running:", cmd if shell else " ".join(map(str, cmd)))
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=cwd, shell=shell)
    if check and r.returncode != 0:
        sys.exit(f"command failed ({r.returncode}):\n{r.stderr}\n{r.stdout}")
    return r


# ----------------------------------------------------------------------------
# setup-prime : install + test PRIME and MixMHCpred 3.x
# ----------------------------------------------------------------------------
def cmd_setup_prime(args):
    prime = Path(args.prime_dir)
    mix = Path(args.mix_dir)
    for name, path, url in (("PRIME", prime, URL_PRIME), ("MixMHCpred", mix, URL_MIX)):
        if path.exists():
            print(f"{name}: found {path}")
            continue
        if not args.accept_license:
            sys.exit(f"{name} not found at {path}.\n"
                     f"PRIME and MixMHCpred are free for ACADEMIC, NON-COMMERCIAL use only. If you accept their\n"
                     f"licenses (PDFs in each repository), re-run with --accept-license to clone {url},\n"
                     f"or download/unzip them yourself and pass --prime-dir / --mix-dir.")
        _run(["git", "clone", "--depth", "1", url, str(path)])

    # MixMHCpred 3.x is a Python program: no compilation, only Python packages (MAFFT is not needed here)
    pip = [sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "scipy", "logomaker", "matplotlib"]
    r = _run(pip, check=False)
    if r.returncode != 0 and "externally-managed" in (r.stderr or "").lower():
        r = _run(pip + ["--break-system-packages"], check=False)
    if r.returncode != 0:
        print("WARNING: pip install failed; continuing (the self-test below will show if a package is missing).\n",
              (r.stderr or "")[-400:])
    mix_exe = (mix / "MixMHCpred").resolve()
    prime_exe = (prime / "PRIME").resolve()
    for f in (mix_exe, prime_exe):
        if not f.exists():
            sys.exit(f"{f} does not exist. Check --prime-dir / --mix-dir (the folder that CONTAINS the 'PRIME' / "
                     f"'MixMHCpred' script).")
        f.chmod(f.stat().st_mode | 0o111)
    # PRIME: on Linux the C++ part must be compiled (README); a build is shipped but we rebuild if possible
    cc = prime / "lib" / "PRIME.cc"
    if cc.exists() and shutil.which("g++"):
        _run(["g++", "-O3", "PRIME.cc", "-o", "PRIME.x"], cwd=str(prime / "lib"))
    # test with PRIME's own example
    test_in = (prime / "test" / "test.txt").resolve()
    test_out = Path("prime_selftest_out.txt").resolve()
    _run([str(prime_exe), "-i", str(test_in), "-o", str(test_out), "-a", "A0101,A2501,B0801,B1801",
          "-mix", str(mix_exe)], cwd=str(prime.resolve()))
    if not test_out.exists() or test_out.stat().st_size == 0:
        sys.exit("PRIME self-test produced no output. Read the messages above.")
    txt = test_out.read_text()
    if "Score_bestAllele" not in txt:
        sys.exit("PRIME self-test output has no Score_bestAllele column; read the file prime_selftest_out.txt")
    print("\nPRIME + MixMHCpred work. Use these paths:")
    print(f"  --prime-exe {prime_exe}\n  --mix-exe   {mix_exe}")


# ----------------------------------------------------------------------------
# prepare
# ----------------------------------------------------------------------------
def cmd_prepare(args):
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    d = pd.read_csv(resolve_file(args.data, 'external dataset, needed by prepare'))
    n0 = len(d)
    d["Peptide"] = d["Peptide"].astype(str).str.strip().str.upper()
    # same restriction as the manuscript: direct T-cell-assay evidence
    d = d[d["Assay"].astype(str).str.contains("T cell assay", case=False, na=False)]
    d = d[d["Peptide"].map(lambda s: len(s) > 0 and set(s) <= AA)]
    d = d.drop_duplicates("Peptide").reset_index(drop=True)
    d["len"] = d["Peptide"].str.len()
    print(f"rows: {n0} -> {len(d)} after T-cell-assay + validity + dedup filters")
    d.to_csv(out / "ext_clean.csv", index=False)

    c1 = d[d["MHC_Class_Used"].astype(str).str.upper() == "I"]
    prime_in = c1[c1["len"].between(8, 14)]
    iedb_in = c1[c1["len"] == 9]
    prime_in[["Peptide"]].to_csv(out / "prime_input.txt", index=False, header=False)
    iedb_in[["Peptide"]].to_csv(out / "iedb_input_9mer.txt", index=False, header=False)
    for a, g in prime_in.groupby("MHC_Allele_Best"):       # per-allele files for --mode paired
        g[["Peptide"]].to_csv(out / f"prime_input_{prime_allele(a)}.txt", index=False, header=False)
    print(f"all external peptides      : {len(d)}")
    print(f"class I                    : {len(c1)}")
    print(f"  -> PRIME input (8-14 aa) : {len(prime_in)}")
    print(f"  -> IEDB input (9-mers)   : {len(iedb_in)}")
    print(f"class II                   : {(d['MHC_Class_Used'] == 'II').sum()}")
    print(f"files written to {out}/")


# ----------------------------------------------------------------------------
# run-prime
# ----------------------------------------------------------------------------
def _read_prime(path) -> pd.DataFrame:
    """PRIME output: '#' comment lines, then a tab-separated table."""
    df = pd.read_csv(path, sep="\t", comment="#")
    cols = {c.lower(): c for c in df.columns}
    pep = cols.get("peptide")
    score = cols.get("score_bestallele")
    rank = cols.get("%rank_bestallele")
    allele = cols.get("bestallele")
    if pep is None or score is None or rank is None:
        sys.exit(f"Cannot find Peptide / Score_bestAllele / %Rank_bestAllele in {path}. "
                 f"Columns found: {list(df.columns)}. Edit _read_prime().")
    res = df[[pep, score, rank]].rename(columns={pep: "Peptide", score: "prime_score", rank: "prime_rank_pct"})
    if allele:
        res["prime_best_allele"] = df[allele]
    return res


def _prime_cmd(prime_exe, mix_exe, inp, outp, alleles):
    return [str(Path(prime_exe).resolve()), "-i", str(Path(inp).resolve()), "-o", str(Path(outp).resolve()),
            "-a", alleles, "-mix", str(Path(mix_exe).resolve())]


def cmd_run_prime(args):
    out = Path(args.out)
    cwd = str(Path(args.prime_exe).resolve().parent)
    if args.mode == "best":
        # all 6 alleles at once; PRIME reports the best allele per peptide
        alleles = ",".join(prime_allele(a) for a in CLASS_I_ALLELES)
        o = out / "prime_output.txt"
        _run(_prime_cmd(args.prime_exe, args.mix_exe, out / "prime_input.txt", o, alleles), cwd=cwd)
        if not o.exists() or o.stat().st_size == 0:
            sys.exit("PRIME finished but prime_output.txt is missing or EMPTY. Run `setup-prime` first.")
        res = _read_prime(o)
    else:
        parts = []
        for a in CLASS_I_ALLELES:
            f = out / f"prime_input_{prime_allele(a)}.txt"
            if not f.exists():
                continue
            o = out / f"prime_output_{prime_allele(a)}.txt"
            _run(_prime_cmd(args.prime_exe, args.mix_exe, f, o, prime_allele(a)), cwd=cwd)
            parts.append(_read_prime(o))
        res = pd.concat(parts)
    res = res.drop_duplicates("Peptide")
    res.to_csv(out / "prime_scores.csv", index=False)
    print(f"wrote {out / 'prime_scores.csv'}  ({len(res)} peptides, mode={args.mode})")


# ----------------------------------------------------------------------------
# run-iedb : IEDB class I immunogenicity (Calis) through the Next-Generation Tools API
#            (code unchanged from the earlier working version)
# ----------------------------------------------------------------------------
import json  # noqa: E402
import requests  # noqa: E402

BASE = "https://api-nextgen-tools.iedb.org/api/v1"
HEADERS = {"accept": "application/json", "Content-Type": "application/json"}


def build_payload(peptides, allele="HLA-A*02:01", mask_choice="default",
                  length_range=None):
    """One-stage T cell class I job, immunogenicity predictor only.

    `alleles` is a required parameter of the tool. With mask_choice='default'
    the Calis score does not depend on it, so the allele is only a placeholder.
    length_range=None means "score the peptides as given" (no splitting).
    """
    return {
        "pipeline_id": "",
        "run_stage_range": [1, 1],
        "stages": [{
            "stage_number": 1,
            "tool_group": "mhci",
            "input_sequence_text": "\n".join(peptides),
            "input_parameters": {
                "alleles": allele,
                "peptide_length_range": length_range,
                "predictors": [{"type": "immunogenicity",
                                "mask_choice": mask_choice}],
            },
        }],
    }


def submit(payload):
    r = requests.post(f"{BASE}/pipeline", headers=HEADERS,
                      data=json.dumps(payload), timeout=120)
    try:
        resp = r.json()
    except Exception:
        sys.exit(f"Non-JSON response ({r.status_code}): {r.text[:500]}")
    if r.status_code >= 400 or resp.get("errors"):
        sys.exit(f"Submit failed ({r.status_code}): {json.dumps(resp)[:1000]}")
    return resp


def wait_for_result(results_uri, poll=5, timeout=1800):
    t0 = time.time()
    while True:
        r = requests.get(results_uri, headers={"accept": "application/json"},
                         timeout=120)
        res = r.json()
        status = res.get("status")
        if status == "done":
            return res
        if status in ("error", "failed"):
            sys.exit(f"Job failed: {json.dumps(res)[:1000]}")
        if time.time() - t0 > timeout:
            sys.exit("Timeout while waiting for IEDB result")
        print(f"  status={status} ... waiting", flush=True)
        time.sleep(poll)


def parse_result(res) -> pd.DataFrame:
    """Return DataFrame[peptide, score] from the 'peptide_table' result."""
    tables = res["data"]["results"]
    pt = next((t for t in tables if t.get("type") == "peptide_table"), None)
    if pt is None:
        sys.exit("No peptide_table in result. Types: "
                 f"{[t.get('type') for t in tables]}")
    cols = pt["table_columns"]
    names = [c["name"] for c in cols]
    # peptide column
    ip = names.index("peptide")
    # score column of the immunogenicity predictor
    cand = [i for i, c in enumerate(cols)
            if "immunogenicity" in str(c.get("source", "")).lower()
            and "score" in c["name"].lower()]
    if not cand:   # fall back: any score column not from a binding predictor
        cand = [i for i, c in enumerate(cols)
                if "score" in c["name"].lower()
                and "binding" not in str(c.get("source", "")).lower()]
    if not cand:
        sys.exit("Cannot find immunogenicity score column. Columns: "
                 f"{[(c['name'], c.get('source')) for c in cols]}")
    isc = cand[0]
    df = pd.DataFrame({"peptide": [row[ip] for row in pt["table_data"]],
                       "score": [row[isc] for row in pt["table_data"]]})
    df["score"] = pd.to_numeric(df["score"], errors="coerce")
    # one row per peptide (allele placeholder can duplicate rows)
    return df.drop_duplicates("peptide").reset_index(drop=True)


def cmd_run_iedb(args):
    out = Path(args.out)
    peps = [p.strip().upper() for p in open(out / "iedb_input_9mer.txt") if p.strip()]
    peps = list(dict.fromkeys(peps))
    print(f"{len(peps)} unique 9-mers")
    parts = []
    for i in range(0, len(peps), args.chunk):
        chunk = peps[i:i + args.chunk]
        print(f"job {i // args.chunk + 1}: {len(chunk)} peptides")
        resp = submit(build_payload(chunk, args.allele, args.mask))
        res = wait_for_result(resp["results_uri"])
        parts.append(parse_result(res))
    o = pd.concat(parts, ignore_index=True).drop_duplicates("peptide")
    missing = set(peps) - set(o["peptide"])
    if missing:
        print(f"WARNING: {len(missing)} peptides got no score, e.g. {sorted(missing)[:5]}")
    o.to_csv(out / "iedb_output.csv", index=False)
    print(f"saved {len(o)} scores -> {out / 'iedb_output.csv'}")


# ----------------------------------------------------------------------------
# analyze
# ----------------------------------------------------------------------------
def compute_clusters(seqs, threshold=0.80):
    """Union-find clusters, identity = 1 - Levenshtein/max(len) >= threshold (same rule as the main pipeline)."""
    from rapidfuzz import process
    from rapidfuzz.distance import Levenshtein
    n = len(seqs)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    M = process.cdist(seqs, seqs, scorer=Levenshtein.normalized_similarity, score_cutoff=threshold,
                      dtype=np.float32, workers=-1)
    ii, jj = np.where(np.triu(M >= threshold, k=1))
    for i, j in zip(ii, jj):
        ri, rj = find(int(i)), find(int(j))
        if ri != rj:
            parent[ri] = rj
    roots = [find(i) for i in range(n)]
    return pd.factorize(pd.Series(roots))[0]


def paired_cluster_bootstrap(sub, score_cols, clusters, n_boot=2000, seed=42):
    """AUC / PR-AUC percentile CIs and ensemble-minus-score AUC differences. ONE resample is used for every
    score (paired). If `clusters` is given, whole clusters are resampled (similar peptides stay together)."""
    y = sub["label"].to_numpy().astype(int)
    S = {c: sub[c].to_numpy(float) for c in score_cols}
    n = len(sub)
    rng = np.random.RandomState(seed)
    if clusters is not None:
        members = [np.asarray(v) for v in pd.Series(np.arange(n)).groupby(np.asarray(clusters)).groups.values()]
        members = [np.asarray(m) for m in members]
    auc_d = {c: [] for c in score_cols}
    pr_d = {c: [] for c in score_cols}
    for _ in range(n_boot):
        if clusters is None:
            idx = rng.randint(0, n, n)
        else:
            pick = rng.randint(0, len(members), len(members))
            idx = np.concatenate([members[k] for k in pick])
        yb = y[idx]
        if yb.min() == yb.max():
            continue
        for c in score_cols:
            auc_d[c].append(fast_auc(yb, S[c][idx]))
            pr_d[c].append(average_precision_score(yb, S[c][idx]))
    rows = []
    ens_auc = fast_auc(y, S["ens"])
    for c in score_cols:
        auc = fast_auc(y, S[c])
        lo, hi = np.percentile(auc_d[c], [2.5, 97.5])
        plo, phi = np.percentile(pr_d[c], [2.5, 97.5])
        row = {"score": c, "n": n, "n_pos": int(y.sum()), "prevalence": y.mean(), "AUC": auc, "AUC_lo": lo,
               "AUC_hi": hi, "PR_AUC": average_precision_score(y, S[c]), "PR_lo": plo, "PR_hi": phi}
        if c != "ens":
            diff = np.array(auc_d["ens"]) - np.array(auc_d[c])
            dlo, dhi = np.percentile(diff, [2.5, 97.5])
            p = 2 * min((diff <= 0).mean(), (diff >= 0).mean())
            row.update({"ens_minus_score": ens_auc - auc, "diff_lo": dlo, "diff_hi": dhi, "p_boot": min(p, 1.0)})
        if auc < 0.5:
            row["note"] = "AUC<0.5: direction is inverted in this subset (score NOT flipped)"
        rows.append(row)
    return pd.DataFrame(rows)


LABELS = {
    "ens": "Ensemble (frozen model)",
    "rank_score": "NetMHCpan/NetMHCIIpan rank (lower rank = immunogenic)",
    "prime_score": "PRIME Score_bestAllele",
    "prime_rank_score": "PRIME %Rank_bestAllele (lower = immunogenic)",
    "iedb_score": "IEDB immunogenicity (Calis)",
    "short_score": "Baseline: shorter peptide = immunogenic",
    "class_score": "Baseline: predicted class I = immunogenic",
    "rcl_score": "Baseline: rank + class + length (fitted on training set)",
}


def cmd_analyze(args):
    out = Path(args.out)
    if not (out / "ext_clean.csv").exists():
        sys.exit(f"{out / 'ext_clean.csv'} not found. Run the `prepare` step first (same --out folder).")
    args.preds = resolve_file(args.preds, 'ensemble predictions from the external notebook')
    if args.train:
        args.train = resolve_file(args.train, 'training set CSV')
    d = pd.read_csv(out / "ext_clean.csv")
    p = pd.read_csv(args.preds)
    key = "dataset_id" if "dataset_id" in p.columns and "dataset_id" in d.columns else "Peptide"
    if args.proba_col not in p.columns:
        sys.exit(f"'{args.proba_col}' not in {args.preds}; columns: {list(p.columns)}")
    d = d.merge(p[[key, args.proba_col]].rename(columns={args.proba_col: "ens"}), on=key, how="inner")
    print(f"peptides with ensemble predictions: {len(d)}")
    d["label"] = d["label"].astype(int)
    d["MHC_Percentile_Rank"] = pd.to_numeric(d["MHC_Percentile_Rank"], errors="coerce")

    # ---- scores, all oriented so that HIGHER = more immunogenic
    d["rank_score"] = -d["MHC_Percentile_Rank"]
    d["class_score"] = (d["MHC_Class_Used"].astype(str).str.upper() == "I").astype(float)
    sign = -1.0        # shorter peptide = immunogenic (direction seen in the training data)
    tr = None
    if args.train:
        tr = pd.read_csv(args.train)
        tr["len"] = tr["Peptide"].astype(str).str.len()
        a = fast_auc(tr["label"].astype(int).values, -tr["len"].values)
        print(f"training data: AUC of 'shorter peptide' = {a:.3f}")
        if a < 0.5:
            sign = 1.0
            print("  -> in the training data LONGER peptides are immunogenic; direction set accordingly")
        else:
            print("  -> shorter = immunogenic confirmed on the training data")
    else:
        print("WARNING: --train not given; assuming shorter = immunogenic without checking it.")
    d["short_score"] = sign * d["len"].astype(float)

    if tr is not None:                 # 3-column baseline fitted on the TRAINING set only
        def feats(x):
            return np.column_stack([np.log1p(pd.to_numeric(x["MHC_Percentile_Rank"], errors="coerce")),
                                    (x["MHC_Class_Used"].astype(str).str.upper() == "I").astype(float),
                                    x["Peptide"].astype(str).str.len().astype(float)])
        Xtr, ytr = feats(tr), tr["label"].astype(int).values
        ok = ~np.isnan(Xtr).any(1)
        sc = StandardScaler().fit(Xtr[ok])
        lr = LogisticRegression(class_weight="balanced", max_iter=1000).fit(sc.transform(Xtr[ok]), ytr[ok])
        Xe = feats(d)
        d["rcl_score"] = np.where(np.isnan(Xe).any(1), np.nan,
                                  lr.predict_proba(sc.transform(np.nan_to_num(Xe)))[:, 1])

    pf = out / "prime_scores.csv"
    if pf.exists() and pf.stat().st_size > 0:
        pr = pd.read_csv(pf)
        pr["prime_rank_score"] = -pr["prime_rank_pct"]
        d = d.merge(pr[["Peptide", "prime_score", "prime_rank_score"]], on="Peptide", how="left")
        print(f"PRIME scores for {d['prime_score'].notna().sum()} peptides")
    else:
        print("NOTE: prime_scores.csv not found -> PRIME rows will be MISSING from the table (run `run-prime`).")
    ie = out / "iedb_output.csv"
    if ie.exists() and ie.stat().st_size > 0:
        x = pd.read_csv(ie)
        x.columns = [c.strip().lower() for c in x.columns]
        x = x.rename(columns={"peptide": "Peptide", "score": "iedb_score"})[["Peptide", "iedb_score"]]
        d = d.merge(x, on="Peptide", how="left")
        print(f"IEDB scores for {d['iedb_score'].notna().sum()} peptides")
    else:
        print("NOTE: iedb_output.csv not found -> IEDB rows will be MISSING from the table (run `run-iedb`).")

    # ---- clusters for the bootstrap
    clusters = None
    if args.cluster_col and args.cluster_col in d.columns and d[args.cluster_col].nunique() < 0.98 * len(d):
        clusters = d[args.cluster_col].to_numpy()
        print(f"bootstrap resamples '{args.cluster_col}' clusters ({len(np.unique(clusters))} clusters)")
    elif args.compute_clusters:
        t0 = time.time()
        clusters = compute_clusters(d["Peptide"].tolist(), 0.80)
        print(f"computed {len(np.unique(clusters))} clusters at 80% identity ({time.time() - t0:.0f}s)")
    else:
        print("WARNING: no usable cluster column -> peptide-level bootstrap (intervals will be a bit too narrow). "
              "Use --cluster-col similarity_cluster_id or --compute-clusters.")
    d["_cluster"] = clusters if clusters is not None else np.arange(len(d))

    is1 = d["MHC_Class_Used"].astype(str).str.upper() == "I"
    subsets = [
        ("All peptides", d, ["ens", "rank_score", "short_score", "class_score", "rcl_score"]),
        ("Class II", d[~is1], ["ens", "rank_score", "short_score", "rcl_score"]),
        ("Class I (all)", d[is1], ["ens", "rank_score", "short_score", "rcl_score", "prime_score",
                                  "prime_rank_score"]),
        ("Class I 8-14 aa (PRIME input)", d[is1 & d["len"].between(8, 14)],
         ["ens", "rank_score", "prime_score", "prime_rank_score", "short_score", "rcl_score"]),
        ("Class I 9-mers (IEDB input)", d[is1 & (d["len"] == 9)],
         ["ens", "rank_score", "iedb_score", "prime_score", "prime_rank_score", "rcl_score"]),
    ]
    tables = []
    for name, sub, cols in subsets:
        cols = [c for c in cols if c in sub.columns]
        if "prime_score" in cols and "prime_score" not in d.columns:
            cols = [c for c in cols if not c.startswith("prime")]
        sub = sub.dropna(subset=cols)
        cols = [c for c in cols if sub[c].nunique() > 1]       # drop scores that are constant here
        if "ens" not in cols or sub["label"].nunique() < 2 or len(sub) < 20 or len(cols) < 2:
            print(f"skip {name}: not enough data / scores")
            continue
        print(f"bootstrapping {name}: n={len(sub)}, scores={cols}")
        t = paired_cluster_bootstrap(sub, cols, sub["_cluster"].to_numpy() if clusters is not None else None,
                                     n_boot=args.n_boot)
        t.insert(0, "subset", name)
        t["score_label"] = t["score"].map(LABELS)
        tables.append(t)
    res = pd.concat(tables, ignore_index=True)
    res.to_csv(out / "benchmark_table.csv", index=False)
    d.drop(columns=["_cluster"]).to_csv(out / "benchmark_scores_per_peptide.csv", index=False)

    pd.set_option("display.width", 250)
    pd.set_option("display.max_colwidth", 60)
    show = res[["subset", "score_label", "n", "n_pos", "AUC", "AUC_lo", "AUC_hi", "ens_minus_score",
                "diff_lo", "diff_hi", "p_boot"]].round(3)
    txt = show.to_string(index=False)
    notes = ["",
             "HOW TO READ: every score is oriented so that higher = more immunogenic; nothing is flipped after the "
             "fact. 'ens_minus_score' is the ensemble AUC minus the score's AUC, with a paired bootstrap CI; if the CI "
             "contains 0 the difference is not demonstrated.",
             "CAVEATS to state in the paper: (1) the rank is the ensemble's own main input; (2) PRIME and the IEDB "
             "predictor were trained on IEDB-derived data and may have seen external peptides; (3) PRIME and IEDB "
             "are class I only; (4) 'class' is the predicted best class, not an experimental restriction; "
             "(5) the tools predict binding / CD8 immunogenicity, the labels are T-cell assay results.",
             "A score with AUC < 0.5 in a subset means its usual direction is inverted there (e.g. MHC rank in class I)."]
    (out / "benchmark_summary.txt").write_text(txt + "\n" + "\n".join(notes))
    print("\n" + txt + "\n".join(notes))
    print("\nsaved:", out / "benchmark_table.csv", "|", out / "benchmark_summary.txt", "|",
          out / "benchmark_scores_per_peptide.csv")


# ----------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sp = ap.add_subparsers(dest="cmd", required=True)

    a = sp.add_parser("setup-prime", help="install and self-test PRIME + MixMHCpred 3.x")
    a.add_argument("--prime-dir", default="PRIME")
    a.add_argument("--mix-dir", default="MixMHCpred")
    a.add_argument("--accept-license", action="store_true",
                   help="you accept the academic, non-commercial licenses; allows cloning from GitHub")
    a.set_defaults(f=cmd_setup_prime)

    a = sp.add_parser("prepare")
    a.add_argument("--data", required=True)
    a.add_argument("--out", default="bench")
    a.set_defaults(f=cmd_prepare)

    a = sp.add_parser("run-prime")
    a.add_argument("--out", default="bench")
    a.add_argument("--prime-exe", required=True, help="path to the PRIME script, e.g. PRIME/PRIME")
    a.add_argument("--mix-exe", required=True, help="path to the MixMHCpred script, e.g. MixMHCpred-master/MixMHCpred")
    a.add_argument("--mode", choices=["best", "paired"], default="best",
                   help="best: all 6 alleles, best-allele score (default); "
                        "paired: each peptide only against its own predicted allele")
    a.set_defaults(f=cmd_run_prime)

    a = sp.add_parser("run-iedb", help="score the class I 9-mers with the IEDB immunogenicity predictor (needs internet)")
    a.add_argument("--out", default="bench")
    a.add_argument("--chunk", type=int, default=100)
    a.add_argument("--allele", default="HLA-A*02:01")
    a.add_argument("--mask", default="default", choices=["default", "custom", "by_allele"])
    a.set_defaults(f=cmd_run_iedb)

    a = sp.add_parser("analyze")
    a.add_argument("--out", default="bench")
    a.add_argument("--preds", required=True, help="external_validation_predictions.csv")
    a.add_argument("--proba-col", default="predicted_probability")
    a.add_argument("--train", default="", help="training CSV (e.g. train718_with_cluster_groups.csv): checks the "
                                               "length direction and fits the rank+class+length baseline")
    a.add_argument("--cluster-col", default="similarity_cluster_id")
    a.add_argument("--compute-clusters", action="store_true",
                   help="if the cluster column is missing/unusable, compute 80%%-identity clusters (needs rapidfuzz)")
    a.add_argument("--n-boot", type=int, default=2000)
    a.set_defaults(f=cmd_analyze)

    args = ap.parse_args()
    args.f(args)


if __name__ == "__main__":
    main()


## Step 2: Upload the data files


In [ ]:
from google.colab import files
import os

NEEDED = ["external_validation_clean.csv",
          "external_validation_predictions.csv",
          "train718_with_cluster_groups.csv"]

up = files.upload()   # select all three CSVs at once
print("uploaded:", list(up.keys()))

missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print("\n!! These file(s) are still missing:", missing)
    print("   Run this cell again and upload the remaining file(s).")
else:
    print("\nAll files are present. Go to the next step.")

## Step 3: Prepare input files (prepare)


In [ ]:
!python benchmark_tools.py prepare --data external_validation_clean.csv --out bench

## Step 4: PRIME 2.1 (class I)



In [ ]:
!python benchmark_tools.py setup-prime --prime-dir PRIME --mix-dir MixMHCpred --accept-license

In [ ]:
!python benchmark_tools.py run-prime --out bench --prime-exe PRIME/PRIME --mix-exe MixMHCpred/MixMHCpred --mode best

## Step 5: IEDB immunogenicity (Calis), 9-mers only, run automatically from code


In [ ]:
!python benchmark_tools.py run-iedb --out bench

In [ ]:
# View the result
import pandas as pd
x = pd.read_csv('bench/iedb_output.csv')
print(x.shape)
display(x.head())

## Step 6: Analysis (analyze)



In [ ]:
!pip install -q rapidfuzz

In [ ]:
# Before running analyze, check that the required files exist
import os
need = ["external_validation_predictions.csv", "train718_with_cluster_groups.csv", "bench/ext_clean.csv"]
for f in need:
    print(("OK       " if os.path.exists(f) else "MISSING  ") + f)
print("\nCSV files found:", sorted(x for x in os.listdir(".") if x.endswith(".csv")))

In [ ]:
!python benchmark_tools.py analyze --out bench \
    --preds external_validation_predictions.csv \
    --train train718_with_cluster_groups.csv \
    --cluster-col similarity_cluster_id --compute-clusters \
    --n-boot 2000

## Step 7: View and download the result tables

In [ ]:
import pandas as pd
t = pd.read_csv('bench/benchmark_table.csv')
t.round(3)

In [ ]:
from google.colab import files
files.download('bench/benchmark_table.csv')
files.download('bench/benchmark_summary.txt')
files.download('bench/benchmark_scores_per_peptide.csv')